# Doppler Shift Analysis in Absorption Imaging

This report summarizes:
1. The mathematical formulation of the time-varying Doppler shift and scattering rate during absorption imaging.
2. The derivation of the exact RMS displacement from random walk (diffusion) compared to the simple estimation.
3. Numerical simulations demonstrating these conclusions for $^{40}\text{K}$ and $^{87}\text{Rb}$.

## 1. Mathematical Formulation

### 1.1 Time-Varying Doppler Shift
When an atom is exposed to a resonant absorption imaging pulse, it continuously absorbs photons and gains momentum along the beam propagation direction (defined here as the $+z$ direction).

Let:
* $v_z(t)$ be the velocity of the atom along the beam direction at time $t$.
* $v_r = \frac{\hbar k}{m}$ be the recoil velocity.
* $N_{\text{photon}}(t) = \frac{v_z(t)}{v_r}$ be the number of scattered photons at time $t$.

As $v_z(t)$ increases, the laser frequency in the atom's frame is red-shifted by the Doppler shift $\Delta \omega_D(t) = - k v_z(t)$. The time-varying detuning in the atom's frame is:
$$\delta(t) = \delta_0 - k v_z(t)$$

where $\delta_0$ is the initial laser detuning in the lab frame (typically $\delta_0 = 0$ for resonant imaging).

### 1.2 Scattering Rate and Acceleration
The scattering rate $\Gamma_{\text{scat}}$ is given by the two-level system optical Bloch equation:
$$\Gamma_{\text{scat}}(s_0, \delta(t), \gamma) = \frac{\gamma}{2} \frac{s_0}{1 + s_0 + \left(\frac{2 \delta(t)}{\gamma}\right)^2}$$

where:
* $s_0 = I/I_{\text{sat}}$ is the saturation parameter.
* $\gamma$ is the transition linewidth (angular frequency, e.g., $\gamma_{\text{K40, D2}} = 2\pi \times 6.035\text{ MHz}$).

Since each scattered photon transfers a momentum kick of $\hbar k$, the velocity and displacement $r_z(t)$ satisfy the coupled differential equations:
$$\frac{d v_z}{dt} = v_r \Gamma_{\text{scat}}(s_0, \delta(t), \gamma)$$
$$\frac{d r_z}{dt} = v_z(t)$$

## 2. Derivation of RMS Displacement: $r_{\text{rms}}$ vs. $r_{\text{rms, simple}}$

### 2.1 The Simple Estimate ($r_{\text{rms, simple}}$)
The simple estimation assumes that the velocity fluctuations $\Delta v_z$ are fully correlated or that the kicks occur at $t = 0$. Using the standard isotropic random walk spread in 1D:
$$\sigma_{v_z}^2 = \frac{1}{3} v_r^2 N_{\text{photon}}$$

Assuming the fluctuation acts over the entire pulse duration $t_{\text{pulse}}$:
$$r_{\text{rms, simple}} = \sigma_{v_z} t_{\text{pulse}} = \sqrt{\frac{N_{\text{photon}}}{3}} v_r t_{\text{pulse}}$$

### 2.2 The Exact Random Walk Position Variance ($r_{\text{rms}}$)
In reality, the random walk is a diffusion process where photon emission events occur at random times $t_i$. An emission event at time $t_i$ imparts a velocity kick that only has a remaining time $t_{\text{pulse}} - t_i$ to cause a position displacement.

The cumulative position fluctuation is:
$$\Delta r_z(t_{\text{pulse}}) = \sum_{t_i \le t_{\text{pulse}}} \eta_{i, z} (t_{\text{pulse}} - t_i)$$

where $\eta_{i, z}$ is the random velocity kick component along $z$, satisfying $\langle \eta_{i, z}^2 \rangle = \frac{1}{3} v_r^2$. Taking the variance:
$$\langle \Delta r_z^2(t_{\text{pulse}}) \rangle = \frac{1}{3} v_r^2 \sum_{t_i \le t_{\text{pulse}}} (t_{\text{pulse}} - t_i)^2$$

Converting the sum to a time integral using the time-varying scattering rate $\Gamma_{\text{scat}}(t)$:
$$\langle \Delta r_z^2(t_{\text{pulse}}) \rangle = \frac{1}{3} v_r^2 \int_0^{t_{\text{pulse}}} \Gamma_{\text{scat}}(t) (t_{\text{pulse}} - t)^2 \, dt$$

The exact RMS displacement is:
$$r_{\text{rms}} = \sqrt{\langle \Delta r_z^2(t_{\text{pulse}}) \rangle} = \sqrt{\frac{1}{3} v_r^2 \Sigma(t_{\text{pulse}})}$$

where $\Sigma(t_{\text{pulse}}) = \int_0^{t_{\text{pulse}}} \Gamma_{\text{scat}}(t) (t_{\text{pulse}} - t)^2 \, dt$.

### 2.3 Comparison for Constant Scattering Rate
For a constant scattering rate $\Gamma_{\text{scat}}(t) = \Gamma$:
* **Simple Estimate**:
  $$r_{\text{rms, simple}} = \sqrt{\frac{\Gamma t_{\text{pulse}}}{3}} v_r t_{\text{pulse}} = \frac{1}{\sqrt{3}} v_r \sqrt{\Gamma t_{\text{pulse}}^3}$$
* **Exact Integration**:
  $$r_{\text{rms}} = \sqrt{\frac{1}{3} v_r^2 \Gamma \int_0^{t_{\text{pulse}}} (t_{\text{pulse}} - t)^2 \, dt} = \sqrt{\frac{1}{3} v_r^2 \Gamma \frac{t_{\text{pulse}}^3}{3}} = \frac{1}{3} v_r \sqrt{\Gamma t_{\text{pulse}}^3}$$

This reveals a fundamental ratio under a constant scattering rate:
$$\frac{r_{\text{rms, exact}}}{r_{	ext{rms, simple}}} = \frac{1}{\sqrt{3}} \approx 0.577$$

Even with zero Doppler shift, the exact RMS position blur is only **$57.7\%$** of the simple estimate.

In [1]:
import numpy as np
import sys
from pathlib import Path

# Setup paths to import E9 modules
E9path = Path("../../").resolve()
if str(E9path) not in sys.path:
    sys.path.insert(0, str(E9path))

import E9_fn.E9_constants as E9c
import E9_fn.E9_cooltrap as E9ct

In [2]:
def simulate_absorption_imaging_doppler(t_pulse, s0, delta0, gamma, v_r, wavelength):
    """
    Simulates the time-varying Doppler shift and scattering rate during an absorption imaging pulse.
    
    Parameters:
    -----------
    t_pulse : float
        Imaging pulse duration in seconds (e.g., 80e-6).
    s0 : float
        Saturation parameter I/I_sat (dimensionless).
    delta0 : float
        Initial laser detuning in the lab frame (in rad/s, angular frequency).
    gamma : float
        Transition linewidth (in rad/s, angular frequency, e.g., E9c.gamma_K40_D2).
    v_r : float
        Recoil velocity (in m/s, e.g., E9c.v_r767_K40).
    wavelength : float
        Transition wavelength (in m, e.g., E9c.lambda_K40_D2).
        
    Returns:
    --------
    dict
        A dictionary containing:
        - 'v_z_end': Final velocity along the imaging beam (m/s).
        - 'r_z_end': Displacement along the imaging beam (m).
        - 'N_photon': Total number of photons scattered (dimensionless).
        - 'Gamma_avg_in_gamma': Average scattering rate in units of gamma.
        - 'r_rms': Exact RMS displacement from random walk (m).
        - 'r_rms_simple': Simple estimated RMS displacement using final N_photon (m).
        - 'delta_end_in_gamma': Final Doppler shift in units of gamma (linewidth).
    """
    from scipy.integrate import solve_ivp
    
    k = 2 * np.pi / wavelength
    
    def ode_func(t, y):
        # y = [v_z, r_z, sigma]
        v_z = y[0]
        # Detuning including Doppler shift (red shift for positive velocity)
        delta = delta0 - k * v_z
        # Scattering rate Gamma_scat using E9ct.Gamma_scat
        gamma_scat = E9ct.Gamma_scat(s0, delta, gamma)
        
        dv_z_dt = v_r * gamma_scat
        dr_z_dt = v_z
        dsigma_dt = gamma_scat * (t_pulse - t)**2
        
        return [dv_z_dt, dr_z_dt, dsigma_dt]
        
    # Initial conditions: v_z(0) = 0, r_z(0) = 0, sigma(0) = 0
    y0 = [0.0, 0.0, 0.0]
    
    sol = solve_ivp(ode_func, [0, t_pulse], y0, method='RK45', rtol=1e-8, atol=1e-8)
    
    v_z_end = sol.y[0][-1]
    r_z_end = sol.y[1][-1]
    sigma_end = sol.y[2][-1]
    
    N_photon = v_z_end / v_r
    Gamma_avg = N_photon / t_pulse
    
    r_rms = np.sqrt(1/3 * v_r**2 * sigma_end)
    r_rms_simple = np.sqrt(N_photon / 3) * v_r * t_pulse
    
    delta_end_in_gamma = (k * v_z_end) / gamma
    
    return {
        'v_z_end': v_z_end,
        'r_z_end': r_z_end,
        'N_photon': N_photon,
        'Gamma_avg_in_gamma': Gamma_avg / gamma,
        'r_rms': r_rms,
        'r_rms_simple': r_rms_simple,
        'delta_end_in_gamma': delta_end_in_gamma
    }

In [3]:
t_abs = 80e-6
s0_abs = 0.15

res_K40 = simulate_absorption_imaging_doppler(
    t_pulse=t_abs,
    s0=s0_abs,
    delta0=0,
    gamma=E9c.gamma_K40_D2,
    v_r=E9c.v_r767_K40,
    wavelength=E9c.lambda_K40_D2
)

res_Rb87 = simulate_absorption_imaging_doppler(
    t_pulse=t_abs,
    s0=s0_abs,
    delta0=0,
    gamma=E9c.gamma_Rb87_D2,
    v_r=E9c.v_r780_Rb87,
    wavelength=E9c.lambda_Rb87_D2
)

print(f"imaging pulse duration:         {t_abs / 1e-6:.2f} us")
print(f"Imaging beam intensity (s0):    {s0_abs:.2f}")
print("                                            K40       Rb87")
print(f"Scattering rate (avg):                      {res_K40['Gamma_avg_in_gamma']:<10.4f}{res_Rb87['Gamma_avg_in_gamma']:<10.4f} gamma")
print(f"Number of photons scattered by each atom:   {res_K40['N_photon']:<10.4f}{res_Rb87['N_photon']:<10.4f}")
print(f"Velocity along the imaging beam at the end: {res_K40['v_z_end']:<10.4f}{res_Rb87['v_z_end']:<10.4f} m/s")
print(f"Displacement along the imaging beam:        {res_K40['r_z_end'] / 1e-6:<10.4f}{res_Rb87['r_z_end'] / 1e-6:<10.4f} um")
print(f"rms displacement from random walk (exact):  {res_K40['r_rms'] / 1e-6:<10.4f}{res_Rb87['r_rms'] / 1e-6:<10.4f} um")
print(f"rms displacement (simple estimate):         {res_K40['r_rms_simple'] / 1e-6:<10.4f}{res_Rb87['r_rms_simple'] / 1e-6:<10.4f} um")
print(f"Doppler shift at the end:                   {res_K40['delta_end_in_gamma']:<10.4f}{res_Rb87['delta_end_in_gamma']:<10.4f} gamma")

imaging pulse duration:         80.00 us
Imaging beam intensity (s0):    0.15
                                            K40       Rb87
Scattering rate (avg):                      0.0528    0.0614     gamma
Number of photons scattered by each atom:   160.1280  187.1318  
Velocity along the imaging beam at the end: 2.0854    1.1012     m/s
Displacement along the imaging beam:        91.3640   45.3478    um
rms displacement from random walk (exact):  4.6957    2.1914     um
rms displacement (simple estimate):         7.6116    3.7181     um
Doppler shift at the end:                   0.4507    0.2326     gamma
